In [1]:
!pip install ultralytics torchtoolbox bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 24.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from torch.nn import MultiheadAttention
from torchvision import transforms
from PIL import Image
import random
import os
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# ============================================================================
# CONSTANTS
# ============================================================================
CHARACTERS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
SOS_TOKEN = 36
EOS_TOKEN = 37
PAD_TOKEN = len(CHARACTERS) + 2  # = 38
NUM_CLASSES = len(CHARACTERS) + 3  # = 39 (0-35 chars, 36 SOS, 37 EOS, 38 PAD)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def index_to_char(indices, include_special_tokens=False):
    """Chuyển list index thành chuỗi ký tự, dừng lại tại EOS."""
    result = []
    for i in indices:
        i = i.item() if torch.is_tensor(i) else i
        if i == SOS_TOKEN:
            if include_special_tokens:
                result.append('[SOS]')
        elif i == EOS_TOKEN:
            if include_special_tokens:
                result.append('[EOS]')
            break
        elif i == PAD_TOKEN:
            break
        elif 0 <= i < len(CHARACTERS):
            result.append(CHARACTERS[i])
        else:
            if include_special_tokens:
                result.append(f'[UNK_{i}]')
    return ''.join(result)


def char_to_indices(text):
    """Chuyển chuỗi text thành tensor indices với SOS ở đầu, EOS ở cuối."""
    indices = [SOS_TOKEN]
    for c in text:
        if c in CHARACTERS:
            indices.append(CHARACTERS.index(c))
        else:
            indices.append(0)  # Map unknown char to '0'
    indices.append(EOS_TOKEN)
    return torch.tensor(indices, dtype=torch.long)


# Dùng chung cho cả train và val, tránh lặp code
def extract_pred_and_true(pred_indices, true_indices):
    # Pred: cắt tại EOS hoặc PAD (whichever comes first)
    pred_content = []
    for idx in pred_indices:
        if idx == EOS_TOKEN or idx == PAD_TOKEN:
            break
        pred_content.append(idx)
    
    # True: lọc bỏ EOS và PAD, chỉ giữ ký tự thực
    true_content = [x for x in true_indices if x not in (EOS_TOKEN, PAD_TOKEN)]
    
    return pred_content, true_content


def compute_crr(pred_content, true_content):
    total = max(len(pred_content), len(true_content))
    if total == 0:
        return 0, 0
    
    correct = sum(
        p == t for p, t in zip(pred_content, true_content)
    )
    return correct, total


# ============================================================================
# YOLO BACKBONE
# ============================================================================
class YoloBackbone(nn.Module):
    def __init__(self, model_path, target_feature_layer_index=9):
        super().__init__()
        _temp_yolo_instance = YOLO(model_path)
        self.yolo_detection_model = _temp_yolo_instance.model
        self.yolo_detection_model.to(DEVICE)
        self.target_feature_layer_index = target_feature_layer_index

        for param in self.yolo_detection_model.parameters():
            param.requires_grad = True

        self._hook_handle = None
        self._fmap_out_hook = []
        self._register_hook()

    def _hook_fn_extractor(self, module, input_val, output_val):
        if isinstance(output_val, torch.Tensor):
            self._fmap_out_hook.append(output_val)
        elif isinstance(output_val, (list, tuple)):
            for item in output_val:
                if isinstance(item, torch.Tensor):
                    self._fmap_out_hook.append(item)
                    break

    def _register_hook(self):
        layer_to_hook = self.yolo_detection_model.model[self.target_feature_layer_index]
        self._hook_handle = layer_to_hook.register_forward_hook(self._hook_fn_extractor)

    def _remove_hook(self):
        if self._hook_handle:
            self._hook_handle.remove()
            self._hook_handle = None

    def forward(self, x):
        self._fmap_out_hook.clear()
        _ = self.yolo_detection_model(x)
        out_tensor = self._fmap_out_hook[0]
        return out_tensor if out_tensor.dim() == 4 else out_tensor.unsqueeze(0)


# ============================================================================
# RViT (Recurrent Vision Transformer)
# ============================================================================
class RViT(nn.Module):
    def __init__(self, yolo_channels=256, d_model=512, num_patches=1600,
                 n_heads=8, num_encoder_layers=3, dim_feedforward=2048,
                 dropout_rate=0.3, max_seq_length=15):
        super().__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length
        self.export_mode = False
        
        self.proj = nn.Sequential(
            nn.Conv2d(yolo_channels, d_model, kernel_size=3, padding=1),
            nn.BatchNorm2d(d_model),
            nn.ReLU(),
            nn.Dropout2d(dropout_rate)
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout_rate, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model))
        self.region_q = nn.Parameter(torch.zeros(1, 1, d_model))

        self.embed = nn.Embedding(NUM_CLASSES, d_model)
        self.gru_num_layers = 2
        self.gru = nn.GRU(
            d_model, d_model, num_layers=self.gru_num_layers,
            batch_first=True,
            dropout=dropout_rate if self.gru_num_layers > 1 else 0
        )
        self.attn = MultiheadAttention(
            d_model, num_heads=n_heads, batch_first=True, dropout=dropout_rate
        )
        self.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(2 * d_model, NUM_CLASSES)
        )

        self.register_buffer("_sos", torch.tensor(SOS_TOKEN, dtype=torch.long))
        self.register_buffer("_eos", torch.tensor(EOS_TOKEN, dtype=torch.long))

    def _encode(self, fmap):
        b = fmap.size(0)
        x = self.proj(fmap)
        x = x.flatten(2).permute(0, 2, 1)

        n = x.size(1) + 1
        pos = self.pos_embed[:, :n, :]

        q = self.region_q.expand(b, -1, -1)
        x = torch.cat([q, x], dim=1) + pos

        enc = self.encoder(x)
        return enc[:, 0], enc[:, 1:]

    def _decode_step(self, current_token, h, spatial_feats):
        """1 bước decode — dùng chung cho mọi mode."""
        emb = self.embed(current_token).unsqueeze(1)
        g, h = self.gru(emb, h)
        a, _ = self.attn(g, spatial_feats, spatial_feats)
        comb = torch.cat([g.squeeze(1), a.squeeze(1)], dim=-1)
        logits = self.fc(comb)
        return logits, h

    def _decode_train(self, region_feat, spatial_feats, target, teach_ratio):
        b = region_feat.size(0)
        device = region_feat.device
        max_gen_len = target.size(1) - 1

        h = region_feat.unsqueeze(0).repeat(self.gru_num_layers, 1, 1).contiguous()
        current_token = torch.full((b,), SOS_TOKEN, device=device, dtype=torch.long)
        outputs = []

        for t in range(max_gen_len):
            logits, h = self._decode_step(current_token, h, spatial_feats)
            outputs.append(logits)

            if random.random() < teach_ratio:
                current_token = target[:, t + 1]
            else:
                current_token = logits.argmax(-1)

        return torch.stack(outputs, dim=1)

    def _decode_inference(self, region_feat, spatial_feats, forced_output_length=None):
        """Inference bình thường — có early stop."""
        b = region_feat.size(0)
        device = region_feat.device
        max_gen_len = forced_output_length if forced_output_length else (self.max_seq_length - 1)

        h = region_feat.unsqueeze(0).repeat(self.gru_num_layers, 1, 1).contiguous()
        current_token = torch.full((b,), SOS_TOKEN, device=device, dtype=torch.long)
        outputs = []
        finished = torch.zeros(b, dtype=torch.bool, device=device)

        for t in range(max_gen_len):
            logits, h = self._decode_step(current_token, h, spatial_feats)
            outputs.append(logits)

            next_token = logits.argmax(-1)
            finished |= (next_token == EOS_TOKEN)
            current_token = torch.where(
                finished,
                torch.tensor(EOS_TOKEN, device=device, dtype=torch.long),
                next_token
            )
            if finished.all():
                break

        return torch.stack(outputs, dim=1)

    def forward(self, fmap, target=None, teach_ratio=0.5,
                forced_output_length=None):

        region_feat, spatial_feats = self._encode(fmap)

        if self.export_mode:
            return self._decode_export(region_feat, spatial_feats)

        if target is not None:
            return self._decode_train(region_feat, spatial_feats, target, teach_ratio)

        return self._decode_inference(region_feat, spatial_feats, forced_output_length)

    def _decode_export(self, region_feat, spatial_feats):
        """
        ONNX-friendly decode:
        - Loop cố định, không break
        - Không dùng Python bool branching
        - Kết quả GIỐNG HỆT _decode_inference (greedy argmax)
        """
        b = region_feat.size(0)
        device = region_feat.device
        max_steps = self.max_seq_length - 2

        h = region_feat.unsqueeze(0).expand(
            self.gru_num_layers, -1, -1
        ).contiguous()

        current_token = self._sos.expand(b)
        finished = torch.zeros(b, device=device, dtype=torch.float32)
        all_logits = []

        for t in range(max_steps):
            logits, h = self._decode_step(current_token, h, spatial_feats)
            all_logits.append(logits)

            next_token = logits.argmax(dim=-1)
            is_eos = (next_token == self._eos).float()
            finished = torch.clamp(finished + is_eos, max=1.0)

            current_token = torch.where(
                finished > 0.5,
                self._eos.expand(b),
                next_token
            )

        return torch.stack(all_logits, dim=1)
         
    def prepare_export(self):
        self.export_mode = True
        self.eval()
        return self

    def finish_export(self):
        self.export_mode = False
        return self

# ============================================================================
# FULL MODEL (YOLO_RViT)
# ============================================================================
class YOLO_RViT(nn.Module):
    def __init__(self, yolo_path, yolo_target_feature_layer_idx=9, max_seq_length=15):
        super().__init__()
        self.backbone = YoloBackbone(yolo_path, target_feature_layer_index=yolo_target_feature_layer_idx)

        dummy_input = torch.randn(1, 3, 640, 640).to(DEVICE)
        with torch.no_grad():
            dummy_feats = self.backbone(dummy_input)

        yolo_channels = dummy_feats.shape[1]
        h_feat, w_feat = dummy_feats.shape[2], dummy_feats.shape[3]
        num_patches = h_feat * w_feat

        self.rvit = RViT(
            yolo_channels=yolo_channels,
            num_patches=num_patches,
            max_seq_length=max_seq_length
        ).to(DEVICE)

    def forward(self, x, target=None, teach_ratio=0.5, forced_output_length=None):
        x = x.to(DEVICE)
        feats = self.backbone(x)
        return self.rvit(feats, target, teach_ratio, forced_output_length)

# ============================================================================
# DATASET
# ============================================================================
class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, license_dir, max_seq_length=15, is_train=True):
        self.img_dir = img_dir
        self.license_dir = license_dir
        self.max_seq_length = max_seq_length
        self.img_names = [f for f in os.listdir(self.img_dir) if f.endswith('.jpg')]

        if is_train:
            self.transform = transforms.Compose([
                transforms.Resize((640, 640)),
                transforms.RandomApply([
                    transforms.RandomRotation(10),
                    transforms.RandomAffine(degrees=8, translate=(0.03, 0.03), scale=(0.95, 1.05)),
                    transforms.RandomPerspective(distortion_scale=0.05, p=0.3),
                ], p=0.7),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.25),
                transforms.ToTensor(),
                transforms.RandomErasing(p=0.2, scale=(0.01, 0.04), ratio=(0.3, 3.3)),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((640, 640)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)

        license_filename = os.path.splitext(self.img_names[idx])[0] + ".txt"
        license_path = os.path.join(self.license_dir, license_filename)

        with open(license_path, 'r', encoding='utf-8') as f:
            license_text = f.read().upper().strip()

        license_indices = char_to_indices(license_text)
        target = torch.full((self.max_seq_length,), PAD_TOKEN, dtype=torch.long)

        actual_len = min(len(license_indices), self.max_seq_length)
        target[:actual_len] = license_indices[:actual_len]

        return img_tensor, target

    @staticmethod
    def collate_fn(batch):
        images = torch.stack([item[0] for item in batch])
        targets = torch.stack([item[1] for item in batch])
        return images, targets

# ============================================================================
# HYPERPARAMETERS
# ============================================================================
YOLO_MODEL_PATH = '/kaggle/input/models/thittn/t-yolov11s-rodosol/pytorch/default/1/last.pt'
YOLO_TARGET_FEATURE_LAYER_INDEX = 13

IMG_DIR_TRAIN = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/images/train"
LICENSE_DIR_TRAIN = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/text/train"
IMG_DIR_VAL = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/images/val"
LICENSE_DIR_VAL = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/text/val"
IMG_DIR_TEST = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/images/test"
LICENSE_DIR_TEST = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/text/test"

MAX_SEQ_LENGTH = 15
BATCH_SIZE = 32
NUM_WORKERS = 4
LEARNING_RATE = 5e-5
MAX_LR_SCHEDULER = 5e-4
WEIGHT_DECAY = 5e-5
NUM_EPOCHS = 100
ACCUM_STEPS = 2
PATIENCE_EARLY_STOP = 7
TEACH_RATIO_START = 0.7
TEACH_RATIO_END = 0.05
LABEL_SMOOTHING = 0.1

# ============================================================================
# SETUP
# ============================================================================
scaler = torch.amp.GradScaler(DEVICE)
autocast_context = lambda: torch.amp.autocast(DEVICE)

train_dataset_full = LicensePlateDataset(
    img_dir=IMG_DIR_TRAIN, license_dir=LICENSE_DIR_TRAIN,
    max_seq_length=MAX_SEQ_LENGTH, is_train=True
)
val_dataset = LicensePlateDataset(
    img_dir=IMG_DIR_VAL, license_dir=LICENSE_DIR_VAL,
    max_seq_length=MAX_SEQ_LENGTH, is_train=False
)
test_dataset = LicensePlateDataset(
    img_dir=IMG_DIR_TEST, license_dir=LICENSE_DIR_TEST,
    max_seq_length=MAX_SEQ_LENGTH, is_train=False
)
train_dataset = Subset(train_dataset_full, list(range(50)))

train_dataloader = DataLoader(
    train_dataset_full, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda'), drop_last=True
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda')
)

test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda')
)
model = YOLO_RViT(
    YOLO_MODEL_PATH,
    yolo_target_feature_layer_idx=YOLO_TARGET_FEATURE_LAYER_INDEX,
    max_seq_length=MAX_SEQ_LENGTH
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Mỗi accumulation cycle = 1 optimizer step
# Số batch cuối cùng cũng trigger 1 step nếu không chia hết
num_batches = len(train_dataloader)
steps_per_epoch = (num_batches + ACCUM_STEPS - 1) // ACCUM_STEPS

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR_SCHEDULER,
    epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.2,
    div_factor=(MAX_LR_SCHEDULER / LEARNING_RATE) if MAX_LR_SCHEDULER > LEARNING_RATE else 10.0
)
scheduler_type = "OneCycleLR"

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, label_smoothing=LABEL_SMOOTHING)

checkpoint = torch.load("/kaggle/input/models/thittnt/yolo-rvt-v11s-2gru-108epoch/pytorch/default/1/final_yolo_rvit_modelCheckpoint108epoch.pth", map_location=DEVICE,)
model.load_state_dict(checkpoint['model_state_dict'], strict= False)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# ============================================================================
# TRAINING LOOP
# ============================================================================
train_loss_values, val_loss_values = [], []
train_acc_values, val_acc_values, val_acc_sequence_values = [], [], []
epoch_count_list = []
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    epoch_count_list.append(epoch + 1)

    # --- Teach ratio decay ---
    teach_ratio = TEACH_RATIO_START - (TEACH_RATIO_START - TEACH_RATIO_END) * (epoch / max(1, NUM_EPOCHS - 1))

    # ================================================================
    # TRAIN PHASE
    # ================================================================
    model.train()
    train_loss, train_correct, train_total_chars = 0, 0, 0

    pbar_train = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [TRAIN] LR: {optimizer.param_groups[0]['lr']:.2e} "
             f"Teach: {teach_ratio:.2f} Scheduler: {scheduler_type}"
    )

    for batch_idx, (imgs, targets) in enumerate(pbar_train):
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        # --- Forward + Loss ---
        with autocast_context():
            outputs = model(imgs, target=targets, teach_ratio=teach_ratio)
            # outputs: [B, seq_len, NUM_CLASSES], targets[:, 1:]: [B, seq_len]
            flat_outputs = outputs.reshape(-1, NUM_CLASSES)
            flat_targets = targets[:, 1:].reshape(-1)
            loss = loss_fn(flat_outputs, flat_targets)
            loss = loss / ACCUM_STEPS

        # --- Backward ---
        scaler.scale(loss).backward()

        # --- Optimizer step (gradient accumulation) ---
        if (batch_idx + 1) % ACCUM_STEPS == 0 or (batch_idx + 1) == num_batches:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad),
                max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler_type == "OneCycleLR":
                scheduler.step()

        # --- Train metrics (không cần gradient) ---
        train_loss += loss.item() * ACCUM_STEPS
        with torch.no_grad():
            preds = outputs.argmax(-1)
            true_chars = targets[:, 1:]

            for i in range(imgs.size(0)):
                pred_content, true_content = extract_pred_and_true(
                    preds[i].tolist(), true_chars[i].tolist()
                )
                correct, total = compute_crr(pred_content, true_content)
                train_correct += correct
                train_total_chars += total

            # --- Print examples (batch 0 mỗi epoch) ---
            if batch_idx == 0 and imgs.size(0) > 0:
                print("\n--- Training Batch 0 Examples ---")
                for i in range(min(5, imgs.size(0))):
                    pred_example = index_to_char(preds[i].tolist())
                    true_example = index_to_char(true_chars[i].tolist())
                    print(f"  Pred: '{pred_example}'")
                    print(f"  True: '{true_example}'")
                print("-------------------------------")

        pbar_train.set_postfix(loss=loss.item() * ACCUM_STEPS)

    avg_train_loss = train_loss / num_batches if num_batches > 0 else 0
    avg_train_acc = train_correct / train_total_chars if train_total_chars > 0 else 0
    train_loss_values.append(avg_train_loss)
    train_acc_values.append(avg_train_acc)

    # ================================================================
    # VALIDATION PHASE
    # ================================================================
    model.eval()
    val_loss = 0
    val_correct, val_total_chars = 0, 0
    val_correct_sequences, val_total_sequences = 0, 0
    num_samples = 0

    pbar_val = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [VAL]")

    with torch.no_grad():
        for imgs, targets in pbar_val:
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            batch_size = imgs.size(0)
            num_samples += batch_size

            with autocast_context():
                outputs = model(imgs, target=None, teach_ratio=0.0)

            with autocast_context():
                out_seq_len = outputs.size(1)
                tgt_content_len = targets.size(1) - 1  # Bỏ SOS ở đầu target

                # Lấy phần ngắn hơn giữa output và target
                min_len = min(out_seq_len, tgt_content_len)
                outputs_for_loss = outputs[:, :min_len, :]
                targets_for_loss = targets[:, 1:min_len + 1]  # Bỏ SOS, lấy min_len tokens

                flat_outputs_val = outputs_for_loss.reshape(-1, NUM_CLASSES)
                flat_targets_val = targets_for_loss.reshape(-1)
                loss = loss_fn(flat_outputs_val, flat_targets_val)

            val_loss += loss.item()

            preds_val = outputs.argmax(-1) 
            true_chars_val = targets[:, 1:]

            for i in range(batch_size):
                pred_content, true_content = extract_pred_and_true(
                    preds_val[i].tolist(), true_chars_val[i].tolist()
                )

                # CRR
                correct, total = compute_crr(pred_content, true_content)
                val_correct += correct
                val_total_chars += total

                # E2E exact match (toàn bộ biển số phải đúng hoàn toàn)
                if pred_content == true_content:
                    val_correct_sequences += 1
                val_total_sequences += 1

            pbar_val.set_postfix(loss=loss.item())

    # ================================================================
    # EPOCH SUMMARY
    # ================================================================
    avg_val_loss = val_loss / len(val_dataloader) if len(val_dataloader) > 0 else 0
    avg_val_acc = val_correct / val_total_chars if val_total_chars > 0 else 0
    avg_val_sequence_accuracy = val_correct_sequences / val_total_sequences if val_total_sequences > 0 else 0.0

    val_loss_values.append(avg_val_loss)
    val_acc_values.append(avg_val_acc)
    val_acc_sequence_values.append(avg_val_sequence_accuracy)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e} "
          f"| Teach: {teach_ratio:.2f} | Scheduler: {scheduler_type}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train CRR: {avg_train_acc:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val CRR:   {avg_val_acc:.4f}")
    print(f"  Val E2E RR: {avg_val_sequence_accuracy:.4f}")
    print("-" * 70)

    # --- Save best model ---
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        print(f"*** New best CRR: {best_val_acc:.4f}. Saving best_model.pth ***")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'val_crr': avg_val_acc,
            'val_e2e_rr': avg_val_sequence_accuracy,
        }, "best_yolo_rvit_model.pth")

# ================================================================
# TEST PHASE (No gradient, no teacher forcing, pure inference)
# ================================================================
model.eval()
test_correct, test_total_chars = 0, 0
test_correct_sequences, test_total_sequences = 0, 0
test_num_samples = 0
test_predictions = []  # Lưu lại predictions để export nếu cần

pbar_test = tqdm(test_dataloader, desc="[TEST] Evaluating...")

with torch.no_grad():
    for imgs, targets in pbar_test:
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        batch_size = imgs.size(0)
        test_num_samples += batch_size

        with autocast_context():
            # Pure inference: target=None, teach_ratio=0 (no teacher forcing)
            outputs = model(imgs, target=None, teach_ratio=0.0)

        # Compute loss on test (optional, chỉ để tham khảo)
        with autocast_context():
            out_seq_len = outputs.size(1)
            tgt_content_len = targets.size(1) - 1
            min_len = min(out_seq_len, tgt_content_len)
            if min_len > 0:
                outputs_for_loss = outputs[:, :min_len, :]
                targets_for_loss = targets[:, 1:min_len + 1]
                flat_outputs_test = outputs_for_loss.reshape(-1, NUM_CLASSES)
                flat_targets_test = targets_for_loss.reshape(-1)
                loss_test = loss_fn(flat_outputs_test, flat_targets_test)
            else:
                loss_test = torch.tensor(0.0, device=DEVICE)

        preds_test = outputs.argmax(-1) 
        true_chars_test = targets[:, 1:]

        for i in range(batch_size):
            pred_content, true_content = extract_pred_and_true(
                preds_test[i].tolist(), true_chars_test[i].tolist()
            )

            # CRR metric
            correct, total = compute_crr(pred_content, true_content)
            test_correct += correct
            test_total_chars += total

            # E2E exact match
            if pred_content == true_content:
                test_correct_sequences += 1
            test_total_sequences += 1

            # Lưu prediction để phân tích sau (optional)
            test_predictions.append({
                'pred': index_to_char(preds_test[i].tolist()),
                'true': index_to_char(true_chars_test[i].tolist()),
                'match': pred_content == true_content
            })

        pbar_test.set_postfix(loss=loss_test.item() if min_len > 0 else 0.0)

# ================================================================
# TEST SUMMARY
# ================================================================
avg_test_crr = test_correct / test_total_chars if test_total_chars > 0 else 0
avg_test_e2e_rr = test_correct_sequences / test_total_sequences if test_total_sequences > 0 else 0

print("\n" + "=" * 70)
print("🎯 TESTING RESULTS")
print("=" * 70)
print(f"  Test CRR:         {avg_test_crr:.4f}")
print(f"  Test E2E RR:      {avg_test_e2e_rr:.4f}")
print("=" * 70)

# ============================================================================
# SAVE FINAL MODEL
# ============================================================================
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'train_loss_history': train_loss_values,
    'val_loss_history': val_loss_values,
    'train_acc_history': train_acc_values,
    'val_acc_history': val_acc_values,
    'val_acc_sequence_history': val_acc_sequence_values,
}, "final_yolo_rvit_model.pth")

print("\nTraining completed!")
if val_acc_values:
    print(f"Final Val CRR:    {val_acc_values[-1]:.4f}")
if val_acc_sequence_values:
    print(f"Final Val E2E RR: {val_acc_sequence_values[-1]:.4f}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/tmp/ipykernel_23/3079709516.py:152: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
Epoch 1/100 [TRAIN] LR: 5.01e-09 Teach: 0.70 Scheduler: OneCycleLR:   0%|          | 1/250 [00:15<1:02:16, 15.01s/it, loss=3.73]


--- Training Batch 0 Examples ---
  Pred: '91M6371'
  True: 'OYH6371'
  Pred: '86A81422'
  True: 'MSA8I62'
  Pred: '36N6491'
  True: 'MSN6491'
  Pred: '39C08970'
  True: 'ODG0970'
  Pred: '34A04664'
  True: 'QRL0G83'
-------------------------------


Epoch 1/100 [TRAIN] LR: 5.01e-09 Teach: 0.70 Scheduler: OneCycleLR: 100%|██████████| 250/250 [05:38<00:00,  1.35s/it, loss=1.73]
Epoch 1/100 [VAL]: 100%|██████████| 125/125 [01:07<00:00,  1.86it/s, loss=1.52]



Epoch 1/100 | LR: 5.28e-05 | Teach: 0.70 | Scheduler: OneCycleLR
  Train Loss: 2.2751 | Train CRR: 0.4101
  Val Loss:   1.5347 | Val CRR:   0.6600
  Val E2E RR: 0.0843
----------------------------------------------------------------------
*** New best CRR: 0.6600. Saving best_model.pth ***


...


--- Training Batch 0 Examples ---
  Pred: 'OYG5934'
  True: 'OYG5934'
  Pred: 'PPH1653'
  True: 'PPH1653'
  Pred: 'QRE1A05'
  True: 'QRE1A05'
  Pred: 'QRL4J59'
  True: 'QRL4J59'
  Pred: 'QQA4798'
  True: 'QQA4798'
-------------------------------


Epoch 100/100 [TRAIN] LR: 1.95e-07 Teach: 0.05 Scheduler: OneCycleLR: 100%|██████████| 250/250 [05:35<00:00,  1.34s/it, loss=0.896]
Epoch 100/100 [VAL]: 100%|██████████| 125/125 [01:01<00:00,  2.02it/s, loss=0.81]



Epoch 100/100 | LR: 5.01e-09 | Teach: 0.05 | Scheduler: OneCycleLR
  Train Loss: 0.8594 | Train CRR: 0.9392
  Val Loss:   0.8149 | Val CRR:   0.9638
  Val E2E RR: 0.8360
----------------------------------------------------------------------


[TEST] Evaluating...: 100%|██████████| 250/250 [02:21<00:00,  1.77it/s, loss=0.784]



🎯 TESTING RESULTS
  Test CRR:         0.9710
  Test E2E RR:      0.8650

Training completed!
Final Val CRR:    0.9638
Final Val E2E RR: 0.8360


In [3]:
!pip install onnxscript
model.eval()
model.rvit.prepare_export()

# Export toàn bộ model (backbone + encoder + decoder gộp)
dummy_img = torch.randn(1, 3, 640, 640, device=DEVICE)

torch.onnx.export(
    model,
    (dummy_img,),    # Chỉ cần image, export_mode tự chạy decode
    "yolo_rvit_full.onnx",
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={
        "image": {0: "batch"},
        "logits": {0: "batch"},
    },
    opset_version=17,
    do_constant_folding=True,
)

model.rvit.finish_export()
print("Done — yolo_rvit_full.onnx")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 13.0 MB/s eta 0:00:00


/tmp/ipykernel_23/1661836409.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0415 03:12:16.954000 23 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0415 03:12:17.926000 23 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned:

[torch.onnx] Obtain model graph for `YOLO_RViT([...]` with `torch.export.export(..., strict=False)`...


/usr/lib/python3.12/contextlib.py:158: UserWarning: The tensor attributes self.backbone._fmap_out_hook[0], self.backbone.yolo_detection_model.model.23.strides, self.backbone.yolo_detection_model.model.23.anchors, self.rvit.gru._flat_weights[0], self.rvit.gru._flat_weights[1], self.rvit.gru._flat_weights[2], self.rvit.gru._flat_weights[3], self.rvit.gru._flat_weights[4], self.rvit.gru._flat_weights[5], self.rvit.gru._flat_weights[6], self.rvit.gru._flat_weights[7] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  self.gen.throw(value)


[torch.onnx] Obtain model graph for `YOLO_RViT([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `YOLO_RViT([...]` with `torch.export.export(..., strict=True)`...


...

[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Runtim

Applied 149 of general pattern rewrite rules.


Skipping constant folding for op Split with multiple outputs.
Skipping constant folding for op Split with multiple outputs.
Skipping constant folding for op Split with multiple outputs.
Skipping constant folding for op Split with multiple outputs.


Done — yolo_rvit_full.onnx


In [4]:
!pip install tensorrt --break-system-packages
import tensorrt as trt
import os

logger = trt.Logger(trt.Logger.INFO)
builder = trt.Builder(logger)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, logger)

with open("yolo_rvit_full.onnx", "rb") as f:
    if not parser.parse(f.read()):
        for i in range(parser.num_errors):
            print(f"Parse error: {parser.get_error(i)}")
        raise RuntimeError("ONNX parse failed")

print(f"Network inputs: {network.num_inputs}, outputs: {network.num_outputs}")

# Xem input shape thực tế
for i in range(network.num_inputs):
    inp = network.get_input(i)
    print(f"  Input '{inp.name}': {inp.shape}, dtype: {inp.dtype}")

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
config.set_flag(trt.BuilderFlag.FP16)

# ===== THÊM OPTIMIZATION PROFILE =====
profile = builder.create_optimization_profile()
# Batch min=1, opt=1, max=4 (tùy nhu cầu)
profile.set_shape("image", 
    min=(1, 3, 640, 640),
    opt=(1, 3, 640, 640),
    max=(1, 3, 640, 640)
)
config.add_optimization_profile(profile)

print("Building TensorRT engine...")
engine = builder.build_serialized_network(network, config)

if engine is None:
    raise RuntimeError("Build engine failed")

with open("yolo_rvit_full.engine", "wb") as f:
    f.write(engine)

print(f"Saved: yolo_rvit_full.engine ({os.path.getsize('yolo_rvit_full.engine') / 1024 / 1024:.1f} MB)")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.2 MB/s eta 0:00:00
  Created wheel for tensorrt: filename=tensorrt-10.16.1.11-py3-none-any.whl size=16564 sha256=30a616ea03efe1409f8b04244688526433d1f4a1e1b1db3417abcb3a13d52c62
  Stored in directory: /root/.cache/pip/wheels/9d/0c/7c/76b5862f9a4b940416c6277c77feb266b16b842f1cb26d8f9b
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-10.16.1.11-py3-none-any.whl size=23075 sha256=cabea313e0c4d4cea562a1a5a2a4cebf71a9921b6481dc43079b3e69011bbd30
  Stored in directory: /root/.cache/pip/wheels/4f/55/9a/84d81786d3b4213025298a1ed9b

In [5]:
# ============================================================================
# MODEL COMPLEXITY & BENCHMARK (batch_size=1)
# ============================================================================
import numpy as np
from torch.utils.flop_counter import FlopCounterMode

model.eval()

# ============================
# 1. PARAMETER COUNT
# ============================
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone_params = sum(p.numel() for p in model.backbone.parameters())
rvit_params = sum(p.numel() for p in model.rvit.parameters())

# ============================
# 2. MODEL SIZE (FP32 / FP16)
# ============================
model_size_fp32_mb = total_params * 4 / (1024 ** 2)
model_size_fp16_mb = total_params * 2 / (1024 ** 2)

print("=" * 70)
print("MODEL COMPLEXITY")
print("=" * 70)
print(f"  Total params:      {total_params / 1e6:.2f} M")
print(f"  Trainable params:  {trainable_params / 1e6:.2f} M")
print(f"  Backbone (YOLO):   {backbone_params / 1e6:.2f} M")
print(f"  RVT (ViT+GRU):     {rvit_params / 1e6:.2f} M")
print(f"  Model size FP32:   {model_size_fp32_mb:.2f} MB")
print(f"  Model size FP16:   {model_size_fp16_mb:.2f} MB")

# ============================
# 3. FLOPs
# ============================
dummy_input = torch.randn(1, 3, 640, 640).to(DEVICE)

with torch.inference_mode():
    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        _ = model(dummy_input, target=None, teach_ratio=0.0,
                  forced_output_length=MAX_SEQ_LENGTH - 1)
    total_flops = flop_counter.get_total_flops()

print(f"  FLOPs @640x640:    {total_flops / 1e9:.2f} GFLOPs")
print("=" * 70)

# ============================================================================
# BENCHMARK FP32 (batch_size=1)
# ============================================================================
test_dataloader_batch1 = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda')
)

latencies_fp32 = []
NUM_WARMUP = 50

with torch.inference_mode():
    for i, (imgs, targets) in enumerate(tqdm(test_dataloader_batch1, desc="[BENCHMARK FP32]")):
        imgs = imgs.to(DEVICE, non_blocking=True)

        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record()
        outputs = model(imgs, target=None, teach_ratio=0.0)
        end_event.record()
        torch.cuda.synchronize()

        elapsed_ms = start_event.elapsed_time(end_event)
        if i > NUM_WARMUP:
            latencies_fp32.append(elapsed_ms)

latencies_fp32 = np.array(latencies_fp32)

mean_latency_fp32 = np.mean(latencies_fp32)
std_latency_fp32 = np.std(latencies_fp32)
median_latency_fp32 = np.median(latencies_fp32)
fps_fp32 = 1000.0 / mean_latency_fp32

print("\n" + "=" * 70)
print("📊 BENCHMARK RESULTS (FP32, batch_size=1, torch.compile, T4 GPU)")
print("=" * 70)
print(f"  Num samples:      {len(latencies_fp32)} (sau {NUM_WARMUP} warm-up)")
print(f"  Mean latency:     {mean_latency_fp32:.2f} ± {std_latency_fp32:.2f} ms")
print(f"  Median latency:   {median_latency_fp32:.2f} ms")
print(f"  FPS:              {fps_fp32:.1f}")
print("=" * 70)

# ============================
# 4. BENCHMARK FP16
# ============================
model.half()
model = torch.compile(model, mode="reduce-overhead")

benchmark_dataloader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=4, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda')
)

NUM_WARMUP = 50
latencies_fp16 = []

with torch.inference_mode():
    for i, (imgs, targets) in enumerate(tqdm(benchmark_dataloader, desc="[BENCHMARK FP16]")):
        imgs = imgs.to(DEVICE, dtype=torch.float16, non_blocking=True)

        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record()
        outputs = model(imgs, target=None, teach_ratio=0.0)
        end_event.record()
        torch.cuda.synchronize()

        elapsed_ms = start_event.elapsed_time(end_event)
        if i >= NUM_WARMUP:
            latencies_fp16.append(elapsed_ms)

latencies_fp16 = np.array(latencies_fp16)

mean_latency_fp16 = np.mean(latencies_fp16)
std_latency_fp16 = np.std(latencies_fp16)
median_latency_fp16 = np.median(latencies_fp16)
p95_latency_fp16 = np.percentile(latencies_fp16, 95)
fps_fp16 = 1000.0 / mean_latency_fp16

print("\n" + "=" * 70)
print("BENCHMARK RESULTS (FP16, batch_size=1, torch.compile, T4 GPU)")
print("=" * 70)
print(f"  Samples measured:  {len(latencies_fp16)} (after {NUM_WARMUP} warm-up)")
print(f"  Mean latency:      {mean_latency_fp16:.2f} +/- {std_latency_fp16:.2f} ms")
print(f"  Median latency:    {median_latency_fp16:.2f} ms")
print(f"  P95 latency:       {p95_latency_fp16:.2f} ms")
print(f"  FPS (bs=1):        {fps_fp16:.1f}")
print("=" * 70)

MODEL COMPLEXITY
  Total params:      25.15 M
  Trainable params:  25.15 M
  Backbone (YOLO):   9.43 M
  RVT (ViT+GRU):     15.72 M
  Model size FP32:   95.94 MB
  Model size FP16:   47.97 MB
  FLOPs @640x640:    38.70 GFLOPs


[BENCHMARK FP32]: 100%|██████████| 8000/8000 [05:31<00:00, 24.11it/s]



📊 BENCHMARK RESULTS (FP32, batch_size=1, torch.compile, T4 GPU)
  Num samples:      7949 (sau 50 warm-up)
  Mean latency:     40.29 ± 2.30 ms
  Median latency:   39.62 ms
  FPS:              24.8


[BENCHMARK FP16]: 100%|██████████| 8000/8000 [03:49<00:00, 34.88it/s]



BENCHMARK RESULTS (FP16, batch_size=1, torch.compile, T4 GPU)
  Samples measured:  7950 (after 50 warm-up)
  Mean latency:      22.89 +/- 2.03 ms
  Median latency:    22.23 ms
  P95 latency:       26.14 ms
  FPS (bs=1):        43.7


In [6]:
import tensorrt as trt
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit

# ============================================================
# Load TRT engine
# ============================================================
runtime = trt.Runtime(trt.Logger(trt.Logger.WARNING))
with open("yolo_rvit_full.engine", "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())
    
context = engine.create_execution_context()
INPUT_NAME = engine.get_tensor_name(0)
OUTPUT_NAME = engine.get_tensor_name(1)
context.set_input_shape(INPUT_NAME, (1, 3, 640, 640))
output_shape = tuple(context.get_tensor_shape(OUTPUT_NAME))
print(f"Input: {INPUT_NAME} | Output: {OUTPUT_NAME} {output_shape}")

d_input = cuda.mem_alloc(1 * 3 * 640 * 640 * 4)
d_output = cuda.mem_alloc(int(np.prod(output_shape)) * 4)
h_output = np.zeros(output_shape, dtype=np.float32)
stream = cuda.Stream()
context.set_tensor_address(INPUT_NAME, int(d_input))
context.set_tensor_address(OUTPUT_NAME, int(d_output))

def trt_infer(img_np):
    cuda.memcpy_htod_async(d_input, np.ascontiguousarray(img_np).ravel(), stream)
    context.execute_async_v3(stream.handle)
    cuda.memcpy_dtoh_async(h_output, d_output, stream)
    stream.synchronize()
    return h_output.copy()
    
# ============================================================
# Evaluate trên val_dataloader
# ============================================================
test_loader_b1 = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=2, collate_fn=LicensePlateDataset.collate_fn
)
correct_seq, total_seq = 0, 0
correct_chars, total_chars = 0, 0
for i, (img, target) in enumerate(test_loader_b1):
    logits = trt_infer(img.numpy())
    pred_ids = logits[0].argmax(-1).tolist()
    true_ids = target[0, 1:].tolist()
    pred_content, true_content = extract_pred_and_true(pred_ids, true_ids)
    correct, total = compute_crr(pred_content, true_content)
    correct_chars += correct
    total_chars += total
    if pred_content == true_content:
        correct_seq += 1
    total_seq += 1
    if i < 10:
        status = "✓" if pred_content == true_content else "✗"
        print(f"  {status} GT='{index_to_char(true_ids)}' TRT='{index_to_char(pred_ids)}'")
print(f"\n{'='*50}")
print(f"TensorRT FP16 Results:")
print(f"  CRR:          {correct_chars/total_chars:.4f}")
print(f"  E2E Accuracy: {correct_seq/total_seq:.4f} ({correct_seq}/{total_seq})")
print(f"{'='*50}")

# ============================================================
# Benchmark FPS — CUDA Events — batch=1 (dùng test_loader_b1)
# ============================================================
N = 1000
latencies_trt  = []

# 1. Tạo lại DataLoader để đảm bảo chưa bị cạn
benchmark_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=2, collate_fn=LicensePlateDataset.collate_fn
)

# 2. Warmup (50 batch đầu)
for i, (imgs, _) in enumerate(benchmark_loader):
    trt_infer(imgs.numpy().astype(np.float32))
    if i >= 49:
        break

# 3. Benchmark loop
count = 0
for imgs, _ in benchmark_loader:
    
    # Convert torch tensor -> numpy array (float32)
    imgs_np = imgs.numpy().astype(np.float32)
    
    start_event = cuda.Event()
    end_event = cuda.Event()
    
    start_event.record(stream)
    trt_infer(imgs_np)
    end_event.record(stream)
    end_event.synchronize()
    
    latencies_trt .append(start_event.time_till(end_event))
    count += 1

latencies_trt  = np.array(latencies_trt )
actual_N = len(latencies_trt )  # Phòng trường hợp test set < 1000
print(f"\nTensorRT FP16 Speed (batch=1, N={actual_N}):")
print(f"  GPU: {cuda.Device(0).name()}")
print(f"  Mean ± Std: {latencies_trt.mean():.2f} ± {latencies_trt.std():.2f} ms")
print(f"  FPS:        {1000/latencies_trt.mean():.1f}")
print(f"{'='*50}")

[04/15/2026-03:27:59] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
Input: image | Output: logits (1, 13, 39)
  ✓ GT='PPM5173' TRT='PPM5173'
  ✓ GT='QRF8G17' TRT='QRF8G17'
  ✓ GT='PPP9C33' TRT='PPP9C33'
  ✓ GT='MTS7362' TRT='MTS7362'
  ✓ GT='MST6448' TRT='MST6448'
  ✓ GT='OCZ9247' TRT='OCZ9247'
  ✓ GT='QRE1I46' TRT='QRE1I46'
  ✓ GT='OYD5H74' TRT='OYD5H74'
  ✗ GT='ODR4042' TRT='ODR4012'
  ✗ GT='ECF8F62' TRT='GCF8F62'

TensorRT FP16 Results:
  CRR:          0.9708
  E2E Accuracy: 0.8634 (6907/8000)

TensorRT FP16 Speed (batch=1, N=8000):
  GPU: Tesla T4
  Mean ± Std: 6.58 ± 0.32 ms
  FPS:        152.1


In [7]:
# ============================================================================
# 📊 PRINT TABLE METRICS
# ============================================================================

print("\n" + "="*90)
print(f"{'TABLE 1: YOLO-RViT Metrics (640x640, batch=1, T4 GPU)':^90}")
print("="*90)

# Header
print(f"{'Model':<15} {'Params(M)':>10} {'GFLOPs':>10} {'Size(MB) FP32':>14} {'Size(MB) FP16':>14} {'Lat_FP32(ms)':>14} {'FPS_FP32':>10} {'Lat_FP16(ms)':>14} {'FPS_FP16':>10} {'Lat_TRT(ms)':>13} {'FPS_TRT':>10}")
print("-"*90)

# Data row 
print(f"{'YOLO-RViT':<15} "
      f"{total_params/1e6:>10.2f} "
      f"{total_flops/1e9:>10.2f} "
      f"{model_size_fp32_mb:>14.2f} "
      f"{model_size_fp16_mb:>14.2f} "
      f"{mean_latency_fp32:>14.2f} "
      f"{fps_fp32:>10.1f} "
      f"{mean_latency_fp16:>14.2f} "
      f"{fps_fp16:>10.1f} "
      f"{latencies_trt.mean():>13.2f} "
      f"{1000/latencies_trt.mean():>10.1f}")

print("="*90)


                  TABLE 1: YOLO-RViT Metrics (640x640, batch=1, T4 GPU)                   
Model            Params(M)     GFLOPs  Size(MB) FP32  Size(MB) FP16   Lat_FP32(ms)   FPS_FP32   Lat_FP16(ms)   FPS_FP16   Lat_TRT(ms)    FPS_TRT
------------------------------------------------------------------------------------------
YOLO-RViT            25.15      38.70          95.94          47.97          40.29       24.8          22.89       43.7          6.58      152.1
